# 第7回 演習：正則化（解答例・教員用）

## 今日の分析目標

**似た変数だらけでも、係数が信頼できるモデルにしたい。**

この演習では、ブートストラップ（bootstrap）で係数のぐらつきを測り、Ridgeで抑え、Lassoで変数を選び、強さαを交差検証（cross-validation）で選ぶ流れを、自転車データの双子（気温 temp と体感温度 atemp）で自分の手で確かめます。αを動かして「手綱」の感覚をつかみましょう。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
feat = ['temp','atemp','hum','windspeed','season','yr',
        'mnth','holiday','weekday','workingday','weathersit']
X_s = StandardScaler().fit_transform(df[feat].values)
y = df['cnt'].values
print('準備完了（X_s は標準化済み）')

## 1. 多重共線性と係数の不安定性（観察）

まず、そっくりな双子 temp と atemp（相関0.99）を、ブートストラップで観察します。yr（年）が安定し、temp と atemp が暴れる様子を確認してください。

### 深掘り：なぜ双子は「合計だけ」安定するのか

上のセルの結果を、もう一度ゆっくり読み解きます。`yr`（年）の係数はばらつき **30** と安定しているのに、`temp` はばらつき **590**、`atemp` は **602** と、どちらも数百台の幅で毎回暴れています。ところが二つを足した `temp+atemp` の合計は、平均 **950** でほとんど動きません。**片方ずつはグラグラなのに、合計はピタリ**——この一見ふしぎな結果こそ、多重共線性（multicollinearity）の正体です。

**ばらつき（`.std()`）が測っているもの**：ブートストラップは、同じデータから毎回選び直して（重複ありの再抽出）30 通りのデータセットを作り、そのたびに係数を求め直しています。もし係数がデータの小さな入れ替えに過敏なら、30 個の値は大きく散らばります。この散らばりの大きさが「係数のぐらつき（不安定さ）」です。つまり `.std()` は精度ではなく、**この係数をどれだけ信用してよいか**を測る物差しです。

**なぜ合計だけが安定するのか**：`temp` と `atemp` は相関 0.99、ほぼ同じ列です。そのため予測 $X\beta$ は、`temp` と `atemp` を**別々にいくつにするか**にはほとんど左右されず、**二つの係数の合計** $\beta_{\text{temp}}+\beta_{\text{atemp}}$ でほぼ決まります。当てはまりの良さ（誤差）を等高線で描くと、`yr` のような独立な変数の向きには**すり鉢状の深い谷**ができ、底が一点に決まります。ところが双子の向きには、`合計 ≒ 950` を保ったまま「`temp` に多め／`atemp` に多め」と分け方を変えても誤差がほとんど変わらない、**底が平らに伸びた谷（尾根）**ができます。ブートストラップで少しデータが変わるたび、解はこの平らな谷底をスルスル横すべりします。だから**分け方（片方の係数）は暴れ、合計は動かない**のです。

言い換えると、データは「双子あわせた気温の効き」は強く決められるのに、「その手柄を temp と atemp のどちらにどれだけ振るか」は決められない（**識別できない**）。第6回で「額面どおり読めない」と保留した係数の、数値的な原因がこれです。次の節から、この平らな谷にわずかな傾きを与えて**底を一点に定める**道具——正則化（regularization）を入れていきます。

In [ ]:
rng = np.random.default_rng(42); nrow = len(y)
coefs = []
for _ in range(30):
    idx = rng.choice(nrow, nrow, replace=True)
    coefs.append(LinearRegression().fit(X_s[idx], y[idx]).coef_)
coefs = np.array(coefs)
for nm in ['temp', 'atemp', 'yr']:
    i = feat.index(nm)
    print(f'{nm:6s}: 平均{coefs[:,i].mean():+6.0f}  ばらつき{coefs[:,i].std():.0f}')
it, ia = feat.index('temp'), feat.index('atemp')
print(f'temp+atemp の合計: 平均{(coefs[:,it]+coefs[:,ia]).mean():.0f}（合計は安定）')

## 2. Ridge の α を変えて、係数の縮み方を観察する

手綱を強く引いたり、ゆるめたり。αを変えると係数がどう縮むかを、自分の手で確かめます。

### TODO①：αを変えた Ridge の係数比較

αを 1・100・10000 と変えた Ridge を `X_s` と `y` で学習し、temp と atemp の係数がどう変わるかを表示して比べてください。

### 深掘り：Ridge の目的関数と「二乗の罰金」が双子を対等にする理由

**目的関数の骨子**　ふつうの線形（linear）回帰は「誤差だけ」を小さくします。Ridge は、そこに係数への罰金を足した合計を小さくします。

$$
\min_{\beta}\ \underbrace{\lVert y - X\beta \rVert_2^2}_{\text{誤差（当てはまりの悪さ）}}\ +\ \alpha\,\underbrace{\lVert \beta \rVert_2^2}_{\text{罰金（係数の大きさ）}},
\qquad \lVert \beta \rVert_2^2 = \sum_j \beta_j^2
$$

記号を一語ずつほどきます。数式が苦手でも、ここだけ押さえれば十分です。

- $y$：実際に観測された利用台数。$X\beta$：モデルの予測。$\lVert y - X\beta \rVert_2^2$ は「予測の外し」を全データで二乗して足したもの、つまり**誤差**です（小さいほど当てはまりが良い）。
- $\lVert \beta \rVert_2^2 = \sum_j \beta_j^2$：**係数を二乗して全部足した値**。係数が大きいモデルほどこの値が大きくなります。これが**罰金**の本体です。
- $\alpha$（アルファ）：**罰金の重さ＝手綱の強さ**。誤差と罰金のどちらをどれだけ重んじるかを決めるつまみです。$\alpha$ を大きくするほど罰金が効いて係数は強く縮み、$\alpha \to 0$ では罰金が消えてただの最小二乗（線形回帰）に戻ります。

厳密な最小化の導出（正規方程式に $\alpha I$ が足される話など）は、**詳しくは MVA『正則化』回**に譲ります。ここでは「何をしているか」をつかむのが目的です。

**この節の実データで確かめる**　TODO① で $\alpha$ を動かすと、`temp` と `atemp` の係数はおよそ次のように動きます（TODO①を正しく解いたときの実行値）。

| $\alpha$ | temp | atemp | ようす |
|---:|---:|---:|---|
| 1 | +386 | +567 | ほぼ線形回帰。まだちぐはぐ |
| 100 | +455 | +469 | 双子がほぼ対等に |
| 10000 | +75 | +76 | 縮みすぎ（未学習ぎみ） |

手綱をゆるめる（$\alpha$ 小）と暴れたまま、締めすぎる（$\alpha$ 大）と両方つぶれる。**ちょうどよい強さ**が要ります。実際、次節の `RidgeCV` が交差検証で選ぶ値は $\alpha \approx 13.9$ と、比較的ゆるめのあたりに落ち着きます。

**なぜ Ridge は双子を対等にならすのか（二乗の罰金の効き方）**　前の節で、双子は「合計 $\beta_{\text{temp}}+\beta_{\text{atemp}}$ さえ保てば誤差がほぼ変わらない」平らな谷を作っていました。誤差が引き分けなら、**勝負を決めるのは罰金の項**です。合計をある値 $s$ に固定したとき、二乗和 $\beta_{\text{temp}}^2+\beta_{\text{atemp}}^2$ は**二つを等分したとき（$\beta_{\text{temp}}=\beta_{\text{atemp}}=s/2$）に最小**になります。片方に偏せると、大きいほうの二乗が効いて罰金が跳ね上がるからです。

数値で見ると一目瞭然です。線形回帰の分け方 `(371, 582)` は二乗和が $371^2+582^2 \approx 476{,}000$。同じ合計 `953` をきれいに等分した `(476, 477)` なら $\approx 454{,}000$ で、**等分のほうが罰金が軽い**。だから Ridge は谷底を「等分」の側へ寄せ、結果として `temp +455 / atemp +469` とほぼ対等に整理されるのです。平らだった谷に二乗の罰金がわずかな傾きを与え、**底が一点に定まった**——これが「Ridge は多重共線性に強い」の中身です。

**つまずきどころ：なぜ先に標準化（standardization）するのか**　罰金 $\sum_j \beta_j^2$ は**係数の大きさ**に効きます。ところが係数の大きさは変数の単位で変わります（同じ効果でも、単位を1/10にすれば係数は10倍に化ける）。単位を揃えずに罰金をかけると、たまたまスケールの大きい変数だけが得（または損）をする**不公平な手綱**になります。だから正則化の前に必ず標準化して土俵を揃えます。この演習で最初から `X_s`（標準化済み）を使い、上の係数を無単位で比べられるのは、そのためです。

In [ ]:
# TODO: αを 1・100・10000 と変えた Ridge を X_s, y で学習し、temp と atemp の係数を表示して比べてください

# 解答例①：αを変えた Ridge
for a in [1, 100, 10000]:
    m = Ridge(alpha=a).fit(X_s, y)
    print(f'α={a:6d}: temp={m.coef_[it]:+6.0f}  atemp={m.coef_[ia]:+6.0f}')
# → αが大きいほど、係数は小さく縮む


## 3. Lasso でゼロになる変数を数える

Lasso の売りは、いらない変数を自動でゼロにしてくれること。何個ゼロになるか、どの変数が生き残るかを数えます。

### TODO②：ゼロ係数の個数と生き残った変数

α=200 の Lasso を `X_s` と `y` で学習し、ゼロになった係数の個数と、生き残った（ゼロでない）変数名を表示してください。

### 深掘り：なぜ Lasso は係数をピタリとゼロにできるのか（L1 の「角」）

**目的関数の骨子**　Lasso の形は Ridge とそっくりで、**罰金だけが違います**。二乗和のかわりに絶対値の和を使います。

$$
\min_{\beta}\ \lVert y - X\beta \rVert_2^2\ +\ \alpha\,\lVert \beta \rVert_1,
\qquad \lVert \beta \rVert_1 = \sum_j \lvert \beta_j \rvert
$$

$\lVert \beta \rVert_1$ は**係数の絶対値を全部足した値**です。罰金が「二乗和 → 絶対値和」に変わっただけ。でもこのわずかな差が、「全体を縮める」Ridge と「ゼロにする」Lasso という大きな性格の違いを生みます。

**図の言葉で：菱形の「角」で解が軸に乗る**　罰金つきの最小化は、「**罰金の予算を決めておいて**（$\lVert \beta \rVert \le t$ 以内で）、その中でいちばん誤差の小さい係数を選ぶ」問題と同じことです。ここで二つの絵を重ねます。

- **誤差の等高線**：最小二乗解（罰金なしの答え）を中心に、外へ広がる**楕円**。中心から離れるほど誤差が大きい。
- **罰金の予算領域**：Ridge（$\sum \beta_j^2 \le t$）は**円**、Lasso（$\sum \lvert\beta_j\rvert \le t$）は**菱形**（軸上に尖った頂点をもつダイヤ形）。

予算 $t$ を絞っていくと解は領域の**境界**に来ます。答えは「楕円が予算領域に**最初に触れる点**」です。ここで形の違いが効きます。**Lasso の菱形は角が座標軸の上に尖って飛び出している**ので、外から膨らむ楕円は**角に当たりやすい**。角の座標では一方の係数が**ちょうど 0**——つまり Lasso はその変数をモデルから消します。いっぽう **Ridge の円はなめらかで角がない**ので、接点はふつうどの軸上にも乗らず、係数は 0 の**手前で縮むだけ**。これが「Lasso はゼロにする／Ridge は縮めるだけ」の幾何学的な理由です。変数が多い高次元でも同じで、菱形にあたる領域の角や稜が「いくつかの係数が 0」の座標面の上にあるため、Lasso の解は自然と**スパース（sparse、多くがゼロ）**になります。

**同じことを傾きで見る**　0 のすぐそばでの罰金の効き方でも説明できます。絶対値 $\lvert\beta\rvert$ は 0 の近くでも傾きが $\pm\alpha$ で一定——**最後の一歩まで 0 へ押し込む力が残る**。いっぽう二乗の罰金項 $\alpha\beta^2$ の傾きは $2\alpha\beta$ で、0 に近づくほど 0 になり、**押す力が消えて手前で止まる**。菱形の角の話と、この傾きの話は同じ現象の別の見方です。絶対値の点 $\beta=0$ が微分できない（劣微分になる）ことの厳密な扱いは、**詳しくは MVA『正則化』回**へ。

**この節の実データ**　TODO② の α=200 の Lasso では、11 個中 **5 個**の係数がゼロになり、生き残るのは `temp, atemp, windspeed, season, yr, weathersit` の 6 個です（消えるのは湿度 `hum`・月 `mnth`・祝日 `holiday`・曜日 `weekday`・平日 `workingday`）。効きの弱い変数を自動で落とし、年・気温まわり・季節・風速・天気を残す——**変数選択が学習と同時に済む**のが Lasso の売りです。

**つまずきどころ**　ここでは双子の `temp` と `atemp` が両方残りましたが、相関の強い変数が並ぶとき、Lasso が**どちらを残すか**はデータのわずかな違いでぐらつくことがあります。ゼロになった＝無関係、と決めつけるのは早計です。「消えた変数」より「残った変数の顔ぶれ」を、複数の分割で見て確かめる姿勢が安全です。

In [ ]:
# TODO: α=200 の Lasso を X_s, y で学習し、ゼロになった係数の個数と、生き残った変数名を表示してください

# 解答例②：Lasso のゼロ係数
lasso = Lasso(alpha=200, max_iter=50000).fit(X_s, y)
n_zero = np.sum(lasso.coef_ == 0)
alive = [feat[i] for i in range(len(feat)) if lasso.coef_[i] != 0]
print(f'ゼロになった係数: {n_zero} / {len(feat)} 個')
print('生き残った変数:', alive)


## 4. 最適な α を交差検証で選び、双子の決着を見る

RidgeCV で α を自動選択し、Ridge が双子（temp/atemp）をどう整えるかを確認します。

### 深掘り：手綱の強さもデータに選ばせる／U字の先にある「第二の谷」

**$\alpha$ を交差検証で選ぶ**　手綱の強さ $\alpha$ を勘で決めては、せっかくデータに語らせた意味が薄れます。`RidgeCV` は $\alpha$ の候補（ここでは `np.logspace(-1, 4, 50)`＝ $10^{-1}$ から $10^{4}$ まで 50 個）を交差検証で片っ端から試し、**テスト性能がいちばん良い $\alpha$** を選びます。このデータで選ばれるのは $\alpha \approx 13.9$。深掘り②の表でいえば「$\alpha=1$（ほぼ生）と $\alpha=100$（対等）の間」の、比較的ゆるめの手綱です。$\alpha$ という**つまみの位置すらデータに決めさせる**——第5回の交差検証が、ここでそのまま働いています。

**双子の物語の決着**　下のセルの出力で、第1回に仕込み・第6回で暴いた双子が片づきます。線形回帰では `temp +371 / atemp +582` と、いちばん効きそうな気温が体感温度に負けてちぐはぐでした。Ridge をかけると双子はぐっと対等に近づきます。下のセルは見やすさのため強めの α=100 での比較で、そこでは `temp +455 / atemp +469` とほぼ横並び。CV が選ぶ α≈13.9 では `temp +443 / atemp +508` とまだ少し差が残りますが、それでも線形回帰の 371/582 よりずっと対等です。第6回で「額面どおり読めない」と保留した係数が、**安心して読める形**になりました。なお、線形回帰・Ridge・Lasso の予測精度（CV の R²）はどれも 0.78 前後でほぼ同じです——正則化は精度を上げる魔法ではなく、**係数を安定させ解釈を守る**ための道具だ、というのがこの回のいちばんの要点です。

**ひとつ先へ：U字の山の向こうにある第二の谷（double descent）**　第3回では、モデルの自由度（容量）を上げるほどテスト誤差がいったん下がり、やがて過学習（overfitting）で上がる**U字**を見ました。正則化を弱めることは、この容量を上げることと同じ向きの操作です。ところが、容量をさらに上げ続けると話には続きがあります。パラメータ数がデータ数に追いついたあたり（**補間閾値**、パラメータ数 ≈ データ数）でモデルは訓練データを誤差ゼロで通し切れるようになり、テスト誤差はここで**いったん最悪の山**を迎えます。そして**そこを越えてさらに自由度（degrees of freedom）を増やす**と、テスト誤差が**もう一度下がりはじめる**——これが **double descent**（二度目の下降）です。過剰にパラメータがある領域では、訓練データを完全に通す無数の解の中から自然と「係数のノルムが最小」の解が選ばれ、これが**暗黙の正則化**として働くため、と説明されます。

深入りはしません。押さえたいのは、**正則化・容量・過学習は同じ一枚の絵を別の側から見たもの**だということです。手綱をゆるめる＝容量を上げる＝過学習へ近づく、そしてその先にもうひとつ景色がある。理論的な扱いは **詳しくは MVA『正則化』回**へ譲ります。なお今回の自転車データは行数 731・変数 11 と、パラメータ数がデータ数よりずっと少ない「補間閾値（interpolation threshold）のはるか手前」にあります。だから素直な U 字の左側だけを歩いており、第二の谷は現れません。double descent が顔を出すのは、変数やパラメータをデータ数に迫るほど増やした、もっと過剰な設定の話です。この演習では、手綱を強く引いたりゆるめたりして、係数がどう動くかを自分の目でつかんでください。

In [ ]:
rc = RidgeCV(alphas=np.logspace(-1, 4, 50)).fit(X_s, y)
print(f'RidgeCV が選んだ α: {rc.alpha_:.1f}')

ols = LinearRegression().fit(X_s, y)
rdg = Ridge(alpha=100).fit(X_s, y)
print('           temp    atemp')
print(f'線形回帰 : {ols.coef_[it]:+6.0f}  {ols.coef_[ia]:+6.0f}')
print(f'Ridge    : {rdg.coef_[it]:+6.0f}  {rdg.coef_[ia]:+6.0f}  ← 双子がほぼ対等に')

## 目標に答えられたか

- 今日の目標は「似た変数だらけでも、係数が信頼できるモデルにしたい」でした
- 1節で、temp と atemp のばらつきは yr と比べてどうでしたか？ 2つの合計はどうでしたか？
- TODO①で、αを大きくすると temp・atemp の係数はどう動きましたか？
- TODO②で、Lasso は何個の変数をゼロにしましたか？ 生き残ったのはどんな変数でしたか？
- 4節で、Ridge をかけると temp と atemp の係数はどうなりましたか？ 第6回の「額面どおり読めない」問題は解決したでしょうか？

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

3節では α=200 と決め打ちで Lasso を学習しました。今度は α を交差検証で選ばせます。
`LassoCV(cv=5, max_iter=50000)` を `X_s`, `y` で学習し、選ばれた α と、そのときの係数を変数名つきで表示してください。


In [ ]:
from sklearn.linear_model import LassoCV
lcv = LassoCV(cv=5, max_iter=50000).fit(X_s, y)
print(f'選ばれた α = {lcv.alpha_:.1f}')
for name, b in zip(feat, lcv.coef_):
    print(f'{name:12s} {b:+8.1f}')


<details><summary>詰まったら</summary>

4節の `RidgeCV` と同じ使い方です。`from sklearn.linear_model import LassoCV` で読み込み、学習後の `.alpha_` が選ばれた α、`.coef_` が係数です。

</details>


### 応用②（判断）

応用①で選ばれた α のとき、係数がちょうどゼロになった変数は**何個**ですか。整数で答えてください。


In [ ]:
n_zero = int(np.sum(lcv.coef_ == 0))
print('消えた変数:', [feat[i] for i in range(len(feat)) if lcv.coef_[i] == 0])
print('答え:', n_zero)


<details><summary>詰まったら</summary>

`np.sum(lcv.coef_ == 0)` で数えられます（`lcv` は応用①で学習したモデル）。どの変数が消えたかも見ておくと、③で使えます。

</details>


### 応用③（解釈）

双子の temp と atemp のうち、最終的なモデルにどちらを残すべきでしょうか。Ridge（4節）と Lasso（応用①）の結果を根拠に、自転車シェアの運営担当者に向けて3行で説明してください。


**模範例**

Ridge（α≈13.9 で temp +443／atemp +508）も Lasso（α≈37.3 で temp +337／atemp +604）も双子を両方残し、Lasso が消したのは mnth の1つだけでした。データは「気温まわりの効き（合計 約950）」を強く支持しますが、temp と atemp のどちらの手柄かは決めてくれません。

どちらか一方を残すなら temp を勧めます。天気予報で直接手に入り、「気温が1度上がると台数が○台増える」と説明も単純で、atemp は temp などから計算される体感値だからです。

両方入れたまま報告するなら Ridge の係数を使ってください。線形回帰の 371／582 のようなちぐはぐな配分ではなく対等に近い値を安定して返し、4節のとおり CV の R² は 0.78 前後で変わりません。


<details><summary>詰まったら</summary>

Ridge と Lasso は双子をどう扱いましたか（両方残した／片方を消した／対等にした）。データが決めてくれないなら、何を基準に選びますか。

</details>


## 発展（任意）

### double descent を実際に見る

4節の深掘りで「U字の山の向こうにある第二の谷」を予告しました。ここでは、それを自分の手で描きます。

「パラメータが多いほど過学習する」という第3回の常識は、実は途中までしか正しくありません。パラメータ数がデータ数に追いつく点（補間閾値）でテスト誤差は最悪の山を迎え、そこを越えてさらに増やすと**もう一度下がり**ます。

この現象は 2019年ごろに double descent として整理され、パラメータ数がデータ数を桁違いに上回る深層学習が「なぜ過学習で壊れないのか」の理解を進めるきっかけになりました。

自転車データ（731行・11変数）は補間閾値のはるか手前なので、ここでは人工データで再現します。100件の訓練データに対して、特徴の数 p を 5 から 300 まで動かし、テスト誤差の形を見ます。

`LinearRegression` は p > n でも動きます（無数にある「訓練データを完全に通す解」のうち、係数のノルムが最小のものを返します）。


In [ ]:
# 人工データ: 300個の特徴が少しずつ効く線形モデル + ノイズ。訓練100件、テスト500件
rng2 = np.random.default_rng(0)
n_train, n_test, p_max = 100, 500, 300
beta = rng2.standard_normal(p_max) / np.sqrt(np.arange(1, p_max + 1))   # 先頭の特徴ほど効きが強い
beta = beta / np.linalg.norm(beta)                                       # 信号の分散を 1 に揃える
X_tr, X_te = rng2.standard_normal((n_train, p_max)), rng2.standard_normal((n_test, p_max))
y_tr = X_tr @ beta + 0.5 * rng2.standard_normal(n_train)
y_te = X_te @ beta + 0.5 * rng2.standard_normal(n_test)

# 先頭 p 個の特徴だけを使って学習し、テスト RMSE を記録する
ps = np.arange(5, 301, 5)
rmse_ols, rmse_ridge = [], []
for p in ps:
    ols_dd = LinearRegression().fit(X_tr[:, :p], y_tr)
    rmse_ols.append(np.sqrt(np.mean((ols_dd.predict(X_te[:, :p]) - y_te) ** 2)))
    rdg_dd = Ridge(alpha=10).fit(X_tr[:, :p], y_tr)
    rmse_ridge.append(np.sqrt(np.mean((rdg_dd.predict(X_te[:, :p]) - y_te) ** 2)))
rmse_ols, rmse_ridge = np.array(rmse_ols), np.array(rmse_ridge)

plt.semilogy(ps, rmse_ols, 'o-', label='線形回帰（p>n では最小ノルム解）')
plt.semilogy(ps, rmse_ridge, 's-', label='Ridge (α=10)')
plt.axvline(n_train, color='gray', ls='--', label=f'p = n_train = {n_train}')
plt.axhline(0.5, color='gray', ls=':', label='ノイズの大きさ 0.5')
plt.xlabel('特徴の数 p'); plt.ylabel('テスト RMSE（対数軸）'); plt.legend(); plt.show()

left = rmse_ols[ps < n_train]
print(f'p=5:   RMSE {rmse_ols[0]:.2f}')
print(f'第一の谷: p={ps[np.argmin(left)]} で RMSE {left.min():.2f}')
print(f'山:      p={ps[np.argmax(rmse_ols)]} で RMSE {rmse_ols.max():.2f}')
print(f'p=300: RMSE {rmse_ols[-1]:.2f}')
print(f'Ridge の最大 RMSE: {rmse_ridge.max():.2f}（山がならされる）')


**読み方**　線形回帰（丸）の曲線は、p=5 の RMSE 1.09 から p=15 の 0.94 へいったん下がり（第一の谷）、そこから上がって p=100（= 訓練データ数）で 5.61 の山を迎えます。ここが補間閾値で、訓練データをちょうど通し切れるようになった瞬間、係数が暴れてテストでは最悪になります。

ところが p をさらに増やすと誤差は**再び下がり**、p=300 では 1.05 まで戻ります。第3回のU字なら右肩上がりのままのはずの領域で、第二の谷が現れています。これが double descent です。

Ridge（四角、α=10）を重ねると、山がほとんど消えます（最大でも 1.12）。補間閾値で暴れるのは「訓練データを通し切るために係数が巨大になる」からで、手綱をかけた瞬間にそれが抑えられます。正則化・容量・過学習が同じ一枚の絵だという4節の話が、この図に集約されています。

なお、この設定では第二の谷（1.05）は第一の谷（0.94）より浅く、ノイズの大きさ 0.5 にも届きません。「パラメータを増やせば必ず良くなる」のではなく、「山の向こうにも谷がある」と読んでください。理論的な扱いは MVA『正則化』回へ譲ります。

試すなら、`0.5 * rng2.standard_normal(...)` のノイズを 0.1 や 1.0 に変えて、山の高さと第二の谷の深さがどう変わるかを見てみましょう。
